# A2 · B-dot study and simulation (follow-along)

**Phase A Concept and Technology Development › Attitude Control (ADCS)** · PM card TC-ADCS-A2 · workshop WS-ADCS-1

After separating from the rocket, the satellite will be tumbling. In this notebook you simulate how a **B-dot controller** stops that spin.

> 📘 Manual to read alongside: *The B-dot Earth Average Magnetic Field* (the **Manual** tab in Mission Tree)

### This is a follow-along notebook
- All the code is already here. Run the cells from the top with **Shift + Enter** and look at each result.
- When you're done, build the controller yourself in **A2.1 Run detumble sim**.
- Nothing on this page is saved. Use **Download** if you want to keep your changes.

## 1. Spacecraft and orbit
We use the FASTSAT values from the paper. The first run takes a little longer while numpy and matplotlib load.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Spacecraft: FASTSAT (B-dot paper, section V)
J = np.diag([15.9, 16.9, 11.7])   # moment of inertia [kg·m²]
J_INV = np.linalg.inv(J)
M_MAX = 5.0                        # max magnetorquer dipole moment [A·m²]

# Orbit: 650 km altitude, 72° inclination
MU_E = 3.986e14                    # Earth's gravitational parameter [m³/s²]
R = 6371e3 + 650e3                 # orbit radius [m]
INC = np.radians(72)
T_ORBIT = 2 * np.pi * np.sqrt(R**3 / MU_E)   # orbital period [s]
n = 2 * np.pi / T_ORBIT                      # mean motion [rad/s]
MU_F = 7.95e15                     # Earth's magnetic moment constant [T·m³]

print(f"Orbital period: {T_ORBIT / 60:.1f} min")

## 2. Earth's magnetic field
As the satellite orbits, the magnetic field it sees keeps changing. We use the dipole model from equation (1) of the paper.

In [ ]:
def earth_field(t):
    """Earth's magnetic field in the orbit frame [T]: dipole model, paper eq. (1)"""
    return MU_F / R**3 * np.array([
        np.cos(n * t) * np.sin(INC),
        -np.cos(INC),
        2 * np.sin(n * t) * np.sin(INC),
    ])

t = np.linspace(0, T_ORBIT, 400)
B = np.array([earth_field(ti) for ti in t]) * 1e6   # in μT
plt.figure(figsize=(9, 3))
for i, name in enumerate(["x", "y", "z"]):
    plt.plot(t / T_ORBIT, B[:, i], label=name)
plt.xlabel("orbits"); plt.ylabel("magnetic field [uT]"); plt.title("Earth magnetic field over one orbit (paper Fig. 1)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 3. Rotating the attitude
When the satellite turns, the same field points a different way from the satellite's point of view. These helpers do that rotation.

In [ ]:
def skew(w):
    """Cross product as a matrix: skew(a) @ b == np.cross(a, b)"""
    return np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])

def rotate(C, w, dt):
    """Update attitude matrix C after spinning at rate w for dt seconds (Rodrigues' formula)"""
    rate = np.linalg.norm(w)
    if rate < 1e-12:
        return C
    K = skew(w / rate)
    th = rate * dt
    return (np.eye(3) + np.sin(th) * K + (1 - np.cos(th)) * K @ K).T @ C

## 4. The B-dot control law

$$\mathbf{m} = -k\,\dot{\mathbf{B}}$$

- $\dot{\mathbf{B}}$: how fast the measured field is **changing** (that is, how fast the satellite is spinning)
- $\mathbf{m}$: the dipole moment the magnetorquers produce. It creates a torque **against** the spin.
- The magnetorquers can only produce ±5 A·m², so we clip the result with `np.clip`.

In [ ]:
K_GAIN = 1e7

def bdot_control(b_now, b_prev, dt):
    b_dot = (b_now - b_prev) / dt          # rate of change of the field (finite difference, lecture p. 40)
    m = -K_GAIN * b_dot                    # B-dot law
    return np.clip(m, -M_MAX, M_MAX)       # magnetorquer limit

## 5. Simulation
Every second: ① measure the field → ② run the controller → ③ apply the torque → ④ update the rotation.

In [ ]:
def simulate(controller, orbits=3, dt=1.0, w0_deg=(1, 3, 1)):
    """Simulate the spacecraft spin with controller(b_now, b_prev, dt) -> dipole moment m.
    Returns (orbits, angular rate [deg/s])."""
    steps = int(orbits * T_ORBIT / dt)
    w = np.radians(w0_deg).astype(float)   # tip-off rate after separation (paper: pitch 3°/s, roll and yaw 1°/s)
    C = np.eye(3)                           # starting attitude
    b_prev = C @ earth_field(0)
    ts, ws = np.zeros(steps), np.zeros((steps, 3))
    for i in range(steps):
        t = i * dt
        b = C @ earth_field(t)              # magnetometer reading (body frame)
        m = controller(b, b_prev, dt)       # dipole moment chosen by the controller
        tau = np.cross(m, b)                # magnetic torque τ = m × B
        w = w + J_INV @ (tau - np.cross(w, J @ w)) * dt   # Euler's rotation equation
        C = rotate(C, w, dt)
        b_prev = b
        ts[i], ws[i] = t, w
    return ts / T_ORBIT, np.degrees(ws)

def no_control(b_now, b_prev, dt):
    return np.zeros(3)

def plot_rates(orbits, rates, title):
    fig, ax = plt.subplots(figsize=(9, 3.5))
    for i, name in enumerate(["roll (x)", "pitch (y)", "yaw (z)"]):
        ax.plot(orbits, rates[:, i], label=name, linewidth=1)
    ax.plot(orbits, np.linalg.norm(rates, axis=1), "k--", label="|ω|", linewidth=1.2)
    ax.set_xlabel("orbits")
    ax.set_ylabel("angular rate [deg/s]")
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right")
    plt.show()

## 6. Compare
Compare no control with B-dot control. (This takes a few seconds.)

In [ ]:
orbits, rates = simulate(no_control, orbits=1)
plot_rates(orbits, rates, "No control: still tumbling")

orbits, rates = simulate(bdot_control, orbits=3)
plot_rates(orbits, rates, "B-dot control: rates are damped")
print(f"Start |ω| = {np.linalg.norm(rates[0]):.2f} deg/s → after 3 orbits |ω| = {np.linalg.norm(rates[-1]):.2f} deg/s")

## 7. Think about it
1. Change `K_GAIN` to `1e6` and `3e7` and run again. How does the damping speed change?
2. The rate never quite reaches zero. Why? (Hint: "two revolutions per orbit" in section IV of the paper.)

### Next →
- **A2.1 Run detumble sim**: write `bdot_control` yourself.
- Later, in Phase E: **E1.1 Detumble data** works with gyro data from orbit.